# Watermark U-Net — import, attach weights dataset, run top to bottom

1. Upload this file to Kaggle (New Notebook -> Upload), Settings -> Accelerator -> **GPU**.
2. Add Data -> your `watermark-unet` dataset.
3. Run cells in order. Edit ONLY CELL 0 (LR after the probe). Nothing else needs touching.
Rules: no `&`/`nohup`, no Save-Version mid-train, download `models/*.pt` every few hours.

In [ ]:
# ===== CELL 0 — parameters (edit ONLY here) + GPU + disk gate =====
EPOCHS, BATCH, SIZE, PATIENCE = 80, 12, 384, 15
LR = 2e-4  # set AFTER the probe cell: IoU>0.5 -> 1e-4 | 0.3-0.5 -> 2e-4 | <0.3 -> 3e-4
USE_LOGO = False  # True needs 20GB+ free (15GB download)
N_SYNTH = 5000
MIN_FREE_GB = 8
import shutil
import torch
print('cuda:', torch.cuda.is_available())
free_gb = shutil.disk_usage('/kaggle/working').free / 1e9
print(f'free disk: {free_gb:.1f} GB')
assert torch.cuda.is_available(), 'STOP: no GPU — set Accelerator to GPU first'
assert free_gb >= MIN_FREE_GB, f'STOP: need {MIN_FREE_GB}GB free, have {free_gb:.1f}'
print('gate OK — continue')

In [ ]:
# ===== CELL 1 — repo + deps (re-runnable) =====
import os
os.chdir('/kaggle/working')
get_ipython().system('test -d hamrah-watermark/.git && (cd hamrah-watermark && git pull) || git clone https://github.com/AliTabibAzar/hamrah-watermark.git hamrah-watermark')
os.chdir('/kaggle/working/hamrah-watermark')
get_ipython().system('pip install -q -r requirements-train.txt albumentations rapidocr-onnxruntime gdown')

In [ ]:
# ===== CELL 2 — weights from attached dataset + health verify (STOPS if bad) =====
import glob, os
os.makedirs('models', exist_ok=True)
cands = glob.glob('/kaggle/input/*/watermark-unet*.pt')
assert cands, 'STOP: no weights found — Add Data -> your watermark-unet dataset, then rerun'
get_ipython().system(f'cp "{cands[0]}" models/watermark-unet.pt && ls -la models/watermark-unet.pt')
get_ipython().system('python -c "import torch; sd=torch.load(\'models/watermark-unet.pt\', map_location=\'cpu\'); assert isinstance(sd, dict) and len(sd) > 50; print(\'weights OK,\', len(sd), \'tensors\')"')

In [ ]:
# ===== CELL 3 — CLWD test (skip if converted) =====
import os
need = not (os.path.isdir('data/clwd_test/images') and len(os.listdir('data/clwd_test/images')) > 5000)
print('CLWD-test needed:', need)
if need:
    get_ipython().system('test -f clwd.zip || gdown --fuzzy "https://drive.google.com/file/d/17y1gkUhIV6rZJg1gMG-gzVMnH27fm4Ij/view?usp=sharing" -O clwd.zip')
    get_ipython().system("unrar x clwd.zip 'CLWD/test/*' data/clwd_raw/")
    get_ipython().system('python scripts/convert_clwd.py --src data/clwd_raw/CLWD/test --dst data/clwd_test')
    get_ipython().system('rm -rf data/clwd_raw')
get_ipython().system('ls data/clwd_test/images | wc -l')

In [ ]:
# ===== CELL 4 — CLWD train, ~60K pairs (skip if converted) =====
import os
need = not (os.path.isdir('data/clwd_train/images') and len(os.listdir('data/clwd_train/images')) > 50000)
print('CLWD-train needed:', need)
if need:
    get_ipython().system('test -f clwd.zip || gdown --fuzzy "https://drive.google.com/file/d/17y1gkUhIV6rZJg1gMG-gzVMnH27fm4Ij/view?usp=sharing" -O clwd.zip')
    get_ipython().system("unrar x clwd.zip 'CLWD/train/Mask/*' data/clwd_train_raw/")
    get_ipython().system("unrar x clwd.zip 'CLWD/train/Watermarked_image/*' data/clwd_train_raw/")
    get_ipython().system('python scripts/convert_clwd.py --src data/clwd_train_raw/CLWD/train --dst data/clwd_train')
    get_ipython().system('rm -rf data/clwd_train_raw')
get_ipython().system('ls data/clwd_train/images | wc -l')

In [ ]:
# ===== CELL 5 — synthetic top-up (skip if present) =====
import os
need = not (os.path.isdir('data/synth/images') and len(os.listdir('data/synth/images')) >= 5000)
print('synth needed:', need)
if need:
    get_ipython().system('python scripts/gen_synthetic.py --bg data/clwd_train/images --out data/synth --n 5000')

In [ ]:
# ===== CELL 6 — PROBE current weights (5 min). Set LR in CELL 0 from the table. =====
#   probe IoU > 0.5  -> LR = 1e-4 (fine-tune, do not wreck good weights)
#   probe IoU 0.3-0.5 -> LR = 2e-4
#   probe IoU < 0.3   -> LR = 3e-4
get_ipython().system('python scripts/eval_bulk.py --data data/clwd_test --mode unet --limit 200 --thresholds 0.5 --weights models/watermark-unet.pt')

In [ ]:
# ===== CELL 7 — TRAIN (foreground; refresh-safe via checkpoints) =====
# LR comes from CELL 0 — set it from the CELL 6 probe table FIRST.
# STOP rule: val IoU < 0.3 past epoch 15 -> stop cell, send the log.
# Expect 'resumed from ...' first (else STOP: wrong weights path).
# Download models/watermark-unet.pt every few hours from Output.
get_ipython().system(f'python train_unet.py --data data/clwd_train data/synth --out models --epochs {EPOCHS} --batch {BATCH} --size {SIZE} --lr {LR} --patience {PATIENCE} --scheduler cosine --resume')

In [ ]:
# ===== CELL 8 — eval gate (CLWD must PASS) =====
# Bars: CLWD IoU>=0.45 + recall>=0.70.
get_ipython().system('python scripts/eval_bulk.py --data data/clwd_test --mode ensemble --limit 500 --thresholds 0.3 0.5 0.7 --weights models/watermark-unet.pt')

In [ ]:
# ===== CELL 9 — export checklist (DO THIS before deleting the notebook) =====
# [ ] models/watermark-unet.pt downloaded AND added to the Kaggle dataset (new version name!)
# [ ] epoch log copied (best val IoU + epoch number)
# [ ] eval GATE line + table sent for review
get_ipython().system('ls -la models/*.pt')